# 🔍 Notebook 04 — Inference
**FracAtlas YOLOv12m Fracture Detection Pipeline**

Run the trained model on new X-ray images and visualise detected fracture bounding boxes.


## 0. Imports

In [ ]:
import os, yaml, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO
from tqdm import tqdm
import cv2

print("✅ Libraries loaded")


## 1. Configuration

In [ ]:
BASE_DIR   = Path(".")
RUNS_DIR   = BASE_DIR / "runs" / "detect"
RUN_NAME   = "fracatlas_yolo12m"
YAML_PATH  = BASE_DIR / "datasets" / "fracatlas.yaml"
BEST_MODEL = RUNS_DIR / RUN_NAME / "weights" / "best.pt"

# ─── Inference settings ───────────────────────────────────────────────────────
# Adjust CONF_THRESHOLD based on use case:
#   Screening (maximize recall) : 0.10 – 0.20
#   Balanced diagnosis          : 0.25 – 0.40
#   High-precision clinical     : ≥ 0.50
CONF_THRESHOLD = 0.25
IOU_THRESHOLD  = 0.5    # NMS IoU threshold
IMG_SIZE       = 1024   # Must match training imgsz

# Output directory for annotated inference images
INFER_OUT = BASE_DIR / "inference_results"
INFER_OUT.mkdir(exist_ok=True)

assert BEST_MODEL.exists(), f"❌ Model not found: {BEST_MODEL}"
print(f"✅ Model path  : {BEST_MODEL}")
print(f"   Conf thresh : {CONF_THRESHOLD}")
print(f"   IoU thresh  : {IOU_THRESHOLD}")


## 2. Load Model

In [ ]:
model = YOLO(str(BEST_MODEL))

# Optional: display model summary
print("Model loaded successfully.")
print(f"Classes: {model.names}")


## 3. Inference on Test Set Samples

In [ ]:
with open(YAML_PATH) as f:
    data_cfg = yaml.safe_load(f)

test_img_dir = Path(data_cfg["path"]) / "images" / "test"
test_lbl_dir = Path(data_cfg["path"]) / "labels" / "test"

all_test_imgs = sorted(test_img_dir.iterdir())

# Select a balanced sample: half fractured, half not
frac_imgs    = [p for p in all_test_imgs if (test_lbl_dir / (p.stem+".txt")).exists()
                and (test_lbl_dir / (p.stem+".txt")).stat().st_size > 0]
non_frac_imgs = [p for p in all_test_imgs if p not in frac_imgs]

random.seed(42)
sample_imgs = (random.sample(frac_imgs, min(8, len(frac_imgs))) +
               random.sample(non_frac_imgs, min(4, len(non_frac_imgs))))
random.shuffle(sample_imgs)

print(f"Selected {len(sample_imgs)} test images for visual inference")
print(f"  ({min(8,len(frac_imgs))} fractured + {min(4,len(non_frac_imgs))} non-fractured)")


In [ ]:
def draw_boxes(img_path: Path, predictions, gt_lbl_path: Path = None,
               conf_thresh: float = 0.25) -> np.ndarray:
    """Draw predicted (red) and ground truth (green) bounding boxes on image."""
    img = Image.open(img_path).convert("RGB")
    W, H = img.size
    draw = ImageDraw.Draw(img)

    # ── Ground truth boxes (green) ──
    if gt_lbl_path and gt_lbl_path.exists():
        for line in gt_lbl_path.read_text().strip().splitlines():
            if not line.strip(): continue
            _, cx, cy, w, h = map(float, line.split())
            x1 = (cx - w/2)*W; y1 = (cy - h/2)*H
            x2 = (cx + w/2)*W; y2 = (cy + h/2)*H
            draw.rectangle([x1,y1,x2,y2], outline="lime", width=3)
            draw.text((x1+2, y1+2), "GT", fill="lime")

    # ── Predicted boxes (red) ──
    if predictions and predictions[0].boxes is not None:
        for box in predictions[0].boxes:
            conf = float(box.conf)
            if conf < conf_thresh: continue
            x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()
            draw.rectangle([x1,y1,x2,y2], outline="red", width=3)
            draw.text((x1+2, max(0,y1-15)), f"fracture {conf:.2f}", fill="red")

    return np.array(img)

# Run inference
preds = model.predict(
    [str(p) for p in sample_imgs],
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    imgsz=IMG_SIZE,
    verbose=False
)

print(f"✅ Inference complete on {len(sample_imgs)} images")


## 4. Visualise Predictions

In [ ]:
cols = 4
rows = (len(sample_imgs) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*5))
axes = axes.flatten() if rows > 1 else (axes if cols == 1 else axes.flatten())

for i, (img_path, pred) in enumerate(zip(sample_imgs, preds)):
    lbl_path = test_lbl_dir / (img_path.stem + ".txt")
    annotated = draw_boxes(img_path, [pred], lbl_path, conf_thresh=CONF_THRESHOLD)
    axes[i].imshow(annotated)

    # Count GT and predicted boxes
    gt_n = len([l for l in lbl_path.read_text().strip().splitlines() if l.strip()]) if lbl_path.exists() else 0
    pred_n = len(pred.boxes) if pred.boxes is not None else 0

    color = "#e74c3c" if gt_n > 0 else "#2ecc71"
    axes[i].set_title(f"GT: {gt_n} box | Pred: {pred_n} box",
                       fontsize=9, color=color, fontweight="bold")
    axes[i].axis("off")

# Hide unused subplots
for j in range(len(sample_imgs), len(axes)):
    axes[j].axis("off")

plt.suptitle(f"Inference Results (conf≥{CONF_THRESHOLD})\n🟩 Green = Ground Truth  |  🔴 Red = Prediction",
             fontsize=13, fontweight="bold")
plt.tight_layout()
out_path = INFER_OUT / "inference_grid.png"
plt.savefig(out_path, dpi=SAVE_DPI if 'SAVE_DPI' in dir() else 150)
plt.show()
print(f"✅ Saved inference grid → {out_path}")


## 5. Single-Image Inference (New Images)

In [ ]:
def predict_single(image_path: str, conf: float = 0.25, iou: float = 0.5,
                   show: bool = True, save_dir: Path = INFER_OUT) -> dict:
    """
    Run fracture detection on a single X-ray image.
    
    Parameters
    ----------
    image_path : path to the X-ray image (.png / .jpg)
    conf       : confidence threshold
    iou        : NMS IoU threshold
    show       : display result inline
    save_dir   : directory to save annotated output
    
    Returns
    -------
    dict with keys: detections (list), annotated_image (np.ndarray)
    """
    img_path = Path(image_path)
    assert img_path.exists(), f"Image not found: {img_path}"

    preds = model.predict(str(img_path), conf=conf, iou=iou, imgsz=IMG_SIZE, verbose=False)
    pred  = preds[0]

    detections = []
    if pred.boxes is not None:
        for box in pred.boxes:
            x1,y1,x2,y2 = box.xyxy[0].cpu().numpy()
            detections.append({
                "x1": float(x1), "y1": float(y1),
                "x2": float(x2), "y2": float(y2),
                "confidence": float(box.conf),
                "class": model.names[int(box.cls)]
            })

    # Draw
    annotated = draw_boxes(img_path, [pred], conf_thresh=conf)

    if show:
        fig, ax = plt.subplots(figsize=(8, 10))
        ax.imshow(annotated)
        ax.set_title(f"{img_path.name}\n{len(detections)} fracture(s) detected  (conf≥{conf})",
                     fontsize=12, fontweight="bold",
                     color="#e74c3c" if detections else "#2ecc71")
        ax.axis("off")
        plt.tight_layout()
        plt.show()

    # Save
    out_path = save_dir / f"pred_{img_path.stem}.png"
    Image.fromarray(annotated).save(out_path)

    print(f"\n{'🦴 FRACTURE DETECTED' if detections else '✅ NO FRACTURE DETECTED'}")
    print(f"Detections: {len(detections)}")
    for d in detections:
        print(f"  conf={d['confidence']:.3f}  bbox=({d['x1']:.0f},{d['y1']:.0f},{d['x2']:.0f},{d['y2']:.0f})")
    print(f"Saved to: {out_path}")

    return {"detections": detections, "annotated_image": annotated}


# ── EXAMPLE USAGE ─────────────────────────────────────────────────────────────
# Replace this path with any X-ray image you want to test:
example_img = sample_imgs[0]   # Using a test set image as demo

result = predict_single(str(example_img), conf=CONF_THRESHOLD, show=True)


## 6. Batch Inference on a Folder

In [ ]:
def batch_inference(input_folder: str, conf: float = 0.25, iou: float = 0.5,
                    save_csv: bool = True) -> pd.DataFrame:
    """
    Run inference on all images in a folder and return a summary DataFrame.
    
    Parameters
    ----------
    input_folder : path to folder containing X-ray images
    conf         : confidence threshold
    iou          : NMS IoU threshold
    save_csv     : whether to save results to CSV
    """
    import pandas as pd

    folder = Path(input_folder)
    imgs   = list(folder.glob("*.png")) + list(folder.glob("*.jpg")) + list(folder.glob("*.jpeg"))
    print(f"Found {len(imgs)} images in {folder}")

    rows = []
    for img_path in tqdm(imgs):
        preds = model.predict(str(img_path), conf=conf, iou=iou, imgsz=IMG_SIZE, verbose=False)
        pred  = preds[0]
        n_det = len(pred.boxes) if pred.boxes is not None else 0
        max_conf = float(pred.boxes.conf.max()) if (pred.boxes is not None and n_det > 0) else 0.0

        # Save annotated image
        annotated = draw_boxes(img_path, [pred], conf_thresh=conf)
        Image.fromarray(annotated).save(INFER_OUT / f"pred_{img_path.name}")

        rows.append({
            "image": img_path.name,
            "fracture_detected": n_det > 0,
            "n_detections": n_det,
            "max_confidence": round(max_conf, 4),
        })

    df = pd.DataFrame(rows)
    if save_csv:
        csv_path = INFER_OUT / "batch_results.csv"
        df.to_csv(csv_path, index=False)
        print(f"\n✅ Results saved → {csv_path}")

    print(f"\nSummary:")
    print(f"  Total images  : {len(df)}")
    print(f"  Fractures found: {df['fracture_detected'].sum()}")
    print(f"  Avg confidence (when detected): {df[df['fracture_detected']]['max_confidence'].mean():.3f}")
    return df


# ── EXAMPLE: batch inference on the test set ──────────────────────────────────
# df_results = batch_inference(str(test_img_dir), conf=CONF_THRESHOLD)
# df_results.head(10)

print("batch_inference() is ready — uncomment the two lines above to run it.")


## 7. Threshold Sensitivity Analysis

In [ ]:
# Show how many fractures are detected at different confidence levels
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]

# Use a subset for speed
subset = random.sample(frac_imgs, min(30, len(frac_imgs)))

results_by_thresh = {}
for t in thresholds:
    preds_t = model.predict([str(p) for p in subset], conf=t, iou=0.5,
                             imgsz=IMG_SIZE, verbose=False)
    detected = sum(1 for p in preds_t if p.boxes is not None and len(p.boxes) > 0)
    results_by_thresh[t] = detected

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar([str(t) for t in thresholds],
       [results_by_thresh[t] for t in thresholds],
       color="#3498db", alpha=0.85)
ax.axhline(len(subset), color="#e74c3c", linestyle="--", label=f"Total GT positives ({len(subset)})")
ax.set_xlabel("Confidence Threshold", fontsize=12)
ax.set_ylabel("Fractures Detected", fontsize=12)
ax.set_title("Detection Count vs Confidence Threshold\n(fractured test subset)", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(INFER_OUT / "threshold_sensitivity.png", dpi=150)
plt.show()
print("✅ Saved threshold_sensitivity.png")
print("\n💡 Recommended thresholds:")
print("   Screening (max recall) : 0.10 – 0.15")
print("   Balanced diagnosis     : 0.20 – 0.30")
print("   High precision clinical: ≥ 0.40")


## ✅ Pipeline Complete

All four notebooks have been executed:

| Notebook | Status |
|---|---|
| 01 Data Preparation | ✅ |
| 02 Training | ✅ |
| 03 Evaluation | ✅ |
| 04 Inference | ✅ |

Inference results are saved in `inference_results/`.
